# 07 - Final Results Comparison

Ce notebook agrege les sorties des notebooks 03-06.

Important:
- Les colonnes `auc_reference` viennent de la reference locale de chaque notebook (baseline ou teacher), pas directement du notebook 02.
- Le notebook 02 sert de baseline unifiee de controle.
- Pour comparer les defenses entre elles, utiliser prioritairement `delta_auc` (et non la valeur absolue d'une baseline d'un autre notebook).
- Les notebooks 03-06 sont configures en scenario `black-box attacker strong` pour obtenir une baseline MIA exploitable avant defense.

In [37]:
import os
import re
from io import StringIO
import nbformat
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [38]:
NOTEBOOKS = {
    'defense1_regularization': '03_transformer_with_defense1_regularization.ipynb',
    'defense2_output_limiting': '04_transformer_with_defense2_output_limiting.ipynb',
    'defense3_differential_privacy': '05_transformer_with_defense3_differential_privacy.ipynb',
    'defense4_distillation': '06_transformer_with_defense4_knowledge_distillation.ipynb',
}

def _detect_pipeline_dir():
    cwd = os.path.abspath(os.getcwd())
    if os.path.basename(cwd) == 'transformer_pipeline':
        return cwd
    alt = os.path.join(cwd, 'transformer_pipeline')
    if os.path.isdir(alt):
        return alt
    return cwd

PIPELINE_DIR = _detect_pipeline_dir()
print('PIPELINE_DIR =', PIPELINE_DIR)

def _resolve_nb_path(rel_path):
    p = os.path.join(PIPELINE_DIR, rel_path)
    if os.path.exists(p):
        return p
    p2 = os.path.abspath(rel_path)
    if os.path.exists(p2):
        return p2
    return p

def _postprocess_table(df):
    # Normalize column names and drop pandas index column if present.
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    if len(df.columns) > 0 and str(df.columns[0]).startswith('Unnamed'):
        col0 = df.columns[0]
        vals = pd.to_numeric(df[col0], errors='coerce')
        if vals.notna().all():
            df = df.drop(columns=[col0])
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='ignore')
    return df

def _read_html_tables_from_cell(cell):
    tables = []
    for out in cell.get('outputs', []):
        data = out.get('data', {})
        html = data.get('text/html')
        if not html:
            continue
        if isinstance(html, list):
            html = ''.join(html)
        try:
            parsed = pd.read_html(StringIO(html))
            for t in parsed:
                tables.append(_postprocess_table(t))
        except Exception:
            pass
    return tables

def _read_text_tables_from_cell(cell):
    tables = []
    for out in cell.get('outputs', []):
        data = out.get('data', {})
        txt = data.get('text/plain')
        if not txt:
            continue
        if isinstance(txt, list):
            txt = ''.join(txt)
        txt = str(txt)
        if '\n' not in txt:
            continue
        try:
            t = pd.read_fwf(StringIO(txt))
            if len(t.columns) >= 2:
                tables.append(_postprocess_table(t))
        except Exception:
            pass
    return tables

def _read_tables_from_cell(cell):
    tables = _read_html_tables_from_cell(cell)
    if tables:
        return tables
    return _read_text_tables_from_cell(cell)

def _extract_test_acc_from_stream(nb):
    pattern = re.compile(r'test_acc(?:_def\d+)?\s*=\s*([0-9]*\.?[0-9]+)')
    for cell in nb.cells:
        if cell.get('cell_type') != 'code':
            continue
        for out in cell.get('outputs', []):
            if out.get('output_type') != 'stream':
                continue
            txt = out.get('text', '')
            if isinstance(txt, list):
                txt = ''.join(txt)
            match = pattern.search(str(txt))
            if match:
                try:
                    return float(match.group(1))
                except Exception:
                    pass
    return None

def _load_notebook(nb_path):
    with open(nb_path, 'r', encoding='utf-8') as f:
        return nbformat.read(f, as_version=4)

def _count_output_cells(nb):
    n = 0
    for cell in nb.cells:
        if cell.get('cell_type') == 'code' and cell.get('outputs'):
            n += 1
    return n

def extract_quick_auc_table_from_nb(nb, robust=False):
    for cell in nb.cells:
        if cell.get('cell_type') != 'code':
            continue
        src = ''.join(cell.get('source', []))
        has_quick = 'Quick AUC summary' in src
        has_robust = 'robust' in src.lower()
        if not has_quick:
            continue
        if robust and not has_robust:
            continue
        if (not robust) and has_robust:
            continue

        tables = _read_tables_from_cell(cell)
        for t in tables:
            cols = [str(c) for c in t.columns]
            if 'attack' in cols and 'delta_auc' in cols:
                return t.copy()
    return None

def extract_model_delta_table_from_nb(nb):
    for cell in nb.cells:
        if cell.get('cell_type') != 'code':
            continue
        tables = _read_tables_from_cell(cell)
        for t in tables:
            cols = [str(c) for c in t.columns]
            if 'delta_test_acc' in cols:
                return t.copy()
    return None

PIPELINE_DIR = c:\Users\khalil\Desktop\stage_pfe\mia-defense-evaluation\transformer_pipeline


In [39]:
all_quick = {}
all_quick_robust = {}
all_model = {}

for name, rel_path in NOTEBOOKS.items():
    path = _resolve_nb_path(rel_path)
    if not os.path.exists(path):
        print(f'[MISSING] {rel_path} (resolved as: {path})')
        continue

    nb = _load_notebook(path)
    output_cells = _count_output_cells(nb)
    if output_cells == 0:
        print(f'[WARN] {rel_path}: aucune sortie persistée dans le fichier .ipynb.')
        print('       Ouvre ce notebook, fais Run All puis Save (Ctrl+S), ensuite relance 07.')
        continue

    q = extract_quick_auc_table_from_nb(nb, robust=False)
    q_robust = extract_quick_auc_table_from_nb(nb, robust=True)
    m = extract_model_delta_table_from_nb(nb)

    if q is None:
        print(f'[WARN] Quick AUC table (standard) introuvable dans {rel_path}.')
    else:
        all_quick[name] = q

    if q_robust is None:
        print(f'[WARN] Quick AUC table (robust) introuvable dans {rel_path}.')
    else:
        all_quick_robust[name] = q_robust

    if m is None and name == 'defense2_output_limiting':
        # Fallback: defense2 often logs test_acc in stdout even if mcmp table was not saved.
        test_acc = _extract_test_acc_from_stream(nb)
        if test_acc is not None:
            m = pd.DataFrame([
                {
                    'model': 'Transformer',
                    'test_acc_baseline': float(test_acc),
                    'test_acc_defense2': float(test_acc),
                    'delta_test_acc': 0.0,
                }
            ])
            print(f'[INFO] Defense2 model table reconstructed from stream test_acc={test_acc:.4f}.')

    if m is None:
        print(f'[WARN] Model delta table introuvable dans {rel_path}.')
    else:
        all_model[name] = m

print('Loaded quick tables (standard):', list(all_quick.keys()))
print('Loaded quick tables (robust):', list(all_quick_robust.keys()))
print('Loaded model tables:', list(all_model.keys()))

Loaded quick tables (standard): ['defense1_regularization', 'defense2_output_limiting', 'defense3_differential_privacy', 'defense4_distillation']
Loaded quick tables (robust): ['defense1_regularization', 'defense2_output_limiting', 'defense3_differential_privacy', 'defense4_distillation']
Loaded model tables: ['defense1_regularization', 'defense2_output_limiting', 'defense3_differential_privacy', 'defense4_distillation']


In [40]:
def _pick_col(cols, preferred):
    for p in preferred:
        if p in cols:
            return p
    return None

rows = []
for defense_name, qdf in all_quick.items():
    cols = [str(c) for c in qdf.columns]
    c_baseline = _pick_col(cols, ['auc_mean_baseline', 'auc_baseline', 'auc_teacher'])
    c_defense = _pick_col(cols, ['auc_mean_defense1', 'auc_mean_defense2', 'auc_mean_defense3', 'auc_mean_defense4', 'auc_defense1', 'auc_defense2', 'auc_defense3', 'auc_defense4'])

    for _, r in qdf.iterrows():
        rows.append({
            'defense': defense_name,
            'attack': r['attack'],
            'auc_reference': float(r[c_baseline]) if c_baseline is not None else float('nan'),
            'auc_defense': float(r[c_defense]) if c_defense is not None else float('nan'),
            'delta_auc': float(r['delta_auc']),
        })

comparison_attacks = pd.DataFrame(rows)
if len(comparison_attacks) == 0:
    print('No attack comparison data available yet. Run notebooks 03..06 first.')
else:
    display(comparison_attacks.sort_values(['attack', 'delta_auc']).round(4))

,defense,attack,auc_reference,auc_defense,delta_auc
5,defense2_output_limiting,logistic,0.5082,0.5111,0.0029
11,defense4_distillation,logistic,0.5082,0.5156,0.0074
0,defense1_regularization,logistic,0.4938,0.5071,0.0133
7,defense3_differential_privacy,logistic,0.4938,0.5412,0.0474
9,defense4_distillation,shadow_meta,0.5229,0.4984,-0.0245
6,defense3_differential_privacy,shadow_meta,0.5121,0.5086,-0.0035
3,defense2_output_limiting,shadow_meta,0.5229,0.5221,-0.0008
2,defense1_regularization,shadow_meta,0.5121,0.5973,0.0852
4,defense2_output_limiting,threshold_loss,0.5120,0.5120,0.0000
10,defense4_distillation,threshold_loss,0.5120,0.5148,0.0028


In [41]:
rows_model = []
for defense_name in NOTEBOOKS.keys():
    if defense_name in all_model and len(all_model[defense_name]) > 0:
        row = all_model[defense_name].iloc[0].to_dict()
    else:
        row = {}
    row['defense'] = defense_name
    rows_model.append(row)

comparison_models = pd.DataFrame(rows_model)
if len(comparison_models) == 0:
    print('No model comparison data available yet. Run notebooks 03..06 first.')
else:
    keep_cols = [
        c for c in [
            'defense',
            'test_acc_baseline',
            'test_acc_teacher',
            'test_acc_defense1',
            'test_acc_defense2',
            'test_acc_defense3',
            'test_acc_student',
            'delta_test_acc',
            'epsilon_target',
        ]
        if c in comparison_models.columns
    ]
    display(comparison_models[keep_cols].round(4))

,defense,test_acc_baseline,test_acc_teacher,test_acc_defense1,test_acc_defense2,test_acc_defense3,test_acc_student,delta_test_acc
0,defense1_regularization,0.9120,NaN,0.9366,NaN,NaN,NaN,0.0246
1,defense2_output_limiting,0.9331,NaN,NaN,0.9331,NaN,NaN,0.0000
2,defense3_differential_privacy,NaN,NaN,NaN,NaN,0.581,NaN,-0.3310
3,defense4_distillation,NaN,0.9331,NaN,NaN,NaN,0.9155,-0.0176


In [ ]:
if len(comparison_attacks) > 0:
    shadow = comparison_attacks[comparison_attacks['attack'] == 'shadow_meta'].copy()
    print('=== Ranking by Shadow Meta delta_auc (lower is better privacy) ===')
    display(shadow.sort_values('delta_auc').round(4))

    # Sanity check: baseline attack strength should be high enough for meaningful defense benchmarking
    if 'auc_reference' in shadow.columns and len(shadow) > 0:
        weak_refs = shadow[shadow['auc_reference'] < 0.55]
        if len(weak_refs) > 0:
            print('[WARN] Certaines references shadow_meta sont < 0.55: interpretation defense a prendre avec prudence.')
            display(weak_refs[['defense', 'auc_reference', 'auc_defense', 'delta_auc']].round(4))
        else:
            print('[OK] References shadow_meta >= 0.55 pour toutes les defenses (benchmark defense pertinent).')

if len(comparison_attacks) > 0:
    print('=== Mean delta_auc per defense (all attacks) ===')
    agg = comparison_attacks.groupby('defense', as_index=False)['delta_auc'].mean().sort_values('delta_auc')
    display(agg.round(4))

=== Ranking by Shadow Meta delta_auc (lower is better privacy) ===


,defense,attack,auc_reference,auc_defense,delta_auc
9,defense4_distillation,shadow_meta,0.5229,0.4984,-0.0245
6,defense3_differential_privacy,shadow_meta,0.5121,0.5086,-0.0035
3,defense2_output_limiting,shadow_meta,0.5229,0.5221,-0.0008
2,defense1_regularization,shadow_meta,0.5121,0.5973,0.0852


=== Mean delta_auc per defense (all attacks) ===


,defense,delta_auc
3,defense4_distillation,-0.0048
1,defense2_output_limiting,0.0007
2,defense3_differential_privacy,0.0314
0,defense1_regularization,0.0393


In [43]:
# Robust adaptive comparison (meta + LiRA + boundary)
rows_robust = []
for defense_name, qdf in all_quick_robust.items():
    cols = [str(c) for c in qdf.columns]
    c_baseline = _pick_col(cols, ['auc_mean_baseline', 'auc_baseline', 'auc_teacher'])
    c_defense = _pick_col(cols, ['auc_mean_defense1', 'auc_mean_defense2', 'auc_mean_defense3', 'auc_mean_defense4', 'auc_defense1', 'auc_defense2', 'auc_defense3', 'auc_defense4'])

    for _, r in qdf.iterrows():
        rows_robust.append({
            'defense': defense_name,
            'attack': r['attack'],
            'auc_reference': float(r[c_baseline]) if c_baseline is not None else float('nan'),
            'auc_defense': float(r[c_defense]) if c_defense is not None else float('nan'),
            'delta_auc': float(r['delta_auc']),
        })

comparison_attacks_robust = pd.DataFrame(rows_robust)
if len(comparison_attacks_robust) == 0:
    print('No robust attack comparison data available yet. Run robust cells in notebooks 03..06 first.')
else:
    print('=== Robust adaptive attack comparison ===')
    display(comparison_attacks_robust.sort_values(['attack', 'delta_auc']).round(4))

    if 'shadow_meta' in comparison_attacks_robust['attack'].values:
        print('=== Robust ranking by shadow_meta delta_auc (lower is better privacy) ===')
        shadow_robust = comparison_attacks_robust[comparison_attacks_robust['attack'] == 'shadow_meta'].copy()
        display(shadow_robust.sort_values('delta_auc').round(4))

=== Robust adaptive attack comparison ===


,defense,attack,auc_reference,auc_defense,delta_auc
1,defense2_output_limiting,shadow_meta,0.5138,0.5096,-0.0042
3,defense4_distillation,shadow_meta,0.5138,0.5146,0.0008
2,defense3_differential_privacy,shadow_meta,0.4950,0.5359,0.0409
0,defense1_regularization,shadow_meta,0.4820,0.5505,0.0685


=== Robust ranking by shadow_meta delta_auc (lower is better privacy) ===


,defense,attack,auc_reference,auc_defense,delta_auc
1,defense2_output_limiting,shadow_meta,0.5138,0.5096,-0.0042
3,defense4_distillation,shadow_meta,0.5138,0.5146,0.0008
2,defense3_differential_privacy,shadow_meta,0.4950,0.5359,0.0409
0,defense1_regularization,shadow_meta,0.4820,0.5505,0.0685
